In [1]:
pip install pydoe

In [2]:
import numpy as np
import jax.numpy as jnp
import jax
from jax import vmap, grad, jit, random
from jax.example_libraries import optimizers
from scipy import optimize
import matplotlib.pyplot as plt
from tqdm import trange
from pydoe import lhs
from functools import partial
import itertools

In [18]:
def wave_kan(layers,A,lb,ub):
    def init_params():
        def initializer(key,d_in,d_out):
            He = jnp.sqrt(2/(d_in+d_out))
            W = random.normal(key,shape=(d_out,d_in))*He
            T = jnp.zeros_like(W)
            S = jnp.ones_like(T)
            return W,T,S
        key,*keys = random.split(random.PRNGKey(1234),len(layers))
        params = list(map(initializer,keys,layers[:-1],layers[1:]))
        params.append(jnp.array([1.]))
        return params

    def FF(H,A):
        return jnp.hstack([jnp.sin(jnp.dot(H,A)),jnp.cos(jnp.dot(H,A))])

    def apply(params,x,y,z,t):
        Hx = jnp.stack([x,y,z])
        Ht = jnp.array([t])
        Hx = 2.*(Hx - lb[:-1])/(ub[:-1] - lb[:-1]) - 1.
        Ht = 2.*(Ht - lb[-1])/(ub[-1] - lb[-1]) - 1.
        Hx = FF(Hx,A)
        H = jnp.concatenate([Ht,Hx],axis=0)
        b = jax.nn.elu(params[-1],0)
        for W,T,S in params[:-1]:
            H = H.reshape(1,-1)
            H = (H - T)/S
            Z = jnp.exp(-0.5*H**2)*jnp.cos(b*H)
            H = jnp.sum(jnp.multiply(W,Z),axis = 1)
        return H
    return init_params, apply

In [48]:
class Sampler:
    def __init__(self,dim,coord,fun):
        self.dim = dim
        self.coord = coord
        self.fun = fun
    def sample(self,N):
        X = self.coord[0:1,:] + (self.coord[1:2,:] - self.coord[0:1,:])*lhs(self.dim,N)
        y = self.fun(X[:,0:1],X[:,1:2],X[:,2:3],X[:,3:4])
        return X,y
class PINN:
    def __init__(self,layers,bc_sampler,ic_sampler,res_sampler,X_train,u_train,sig,lb,ub):
        self.layers = layers

        self.bcx_sampler1 = bc_sampler[0]
        self.bcx_sampler2 = bc_sampler[1]

        self.bcy_sampler1 = bc_sampler[2]
        self.bcy_sampler2 = bc_sampler[3]

        self.bcz_sampler1 = bc_sampler[4]
        self.bcz_sampler2 = bc_sampler[5]

        self.ic_sampler = ic_sampler
        self.res_sampler = res_sampler

        self.X_train = X_train
        self.u_train = u_train


        key1,key2 = random.split(random.PRNGKey(12))

        A = random.normal(key1,shape=(3,layers[0]//2))*sig
        init_params, self.apply = wave_kan(layers,A,lb,ub)
        params = init_params()

        self.opt_init,\
        self.opt_update,\
        self.get_params = optimizers.adam(1e-03)
        self.opt_state = self.opt_init(params)

        self.count = itertools.count()
        self.hist_loss_res = []
        self.hist_loss_ic = []
        self.hist_loss_bc = []
        self.b_hist = []

    @partial(jit,static_argnums=(0,))
    def u_net(self,params,x,y,z,t):
        out = self.apply(params,x,y,z,t)
        return out[0]

    @partial(jit,static_argnums=(0,))
    def res_net(self,params,x,y,z,t):
        u_x = grad(self.u_net,1)(params,x,y,z,t)
        u_y = grad(self.u_net,2)(params,x,y,z,t)

        u_xx = grad(grad(self.u_net,1),1)(params,x,y,z,t)
        u_xy = grad(grad(self.u_net,2),1)(params,x,y,z,t)

        u_xxy = grad(grad(grad(self.u_net,2),1),1)(params,x,y,z,t)
        u_xxz = grad(grad(grad(self.u_net,3),1),1)(params,x,y,z,t)
        u_xyt = grad(grad(grad(self.u_net,4),2),1)(params,x,y,z,t)

        u_xxxxy = grad(grad(grad(grad(grad(self.u_net,2),1),1),1),1)(params,x,y,z,t)
        return 3*u_xxz - 2*u_xyt + u_xxxxy - 2*u_xy*u_xx + 2*u_xy**2 + 2*(u_y - u_x)*u_xxy

    @partial(jit,static_argnums=(0,))
    def u_loss(self,params):
        u_pred = vmap(self.u_net,(None,0,None,None,0))(params,self.X_train[:,0],0.5,0.5,self.X_train[:,1])
        loss = jnp.mean((u_pred.flatten() - self.u_train.flatten())**2)
        return loss

    @partial(jit,static_argnums=(0,))
    def ic_loss(self,params,ic_data):
        X_ic, u_ic = ic_data
        u_pred = vmap(self.u_net,(None,0,0,0,0))(params,X_ic[:,0],X_ic[:,1],X_ic[:,2],X_ic[:,3])
        loss = jnp.mean((u_pred.flatten() - u_ic.flatten())**2)
        return loss

    @partial(jit,static_argnums=(0,))
    def bc_loss(self,params,bcx1_data,bcx2_data,bcy1_data,bcy2_data,bcz1_data,bcz2_data):
        X_bc1, u_bcx1 = bcx1_data
        X_bc2, u_bcx2 = bcx2_data

        Y_bc1, u_bcy1 = bcy1_data
        Y_bc2, u_bcy2 = bcy2_data

        Z_bc1, u_bcz1 = bcz1_data
        Z_bc2, u_bcz2 = bcz2_data


        X_bc = jnp.vstack([X_bc1, X_bc2, Y_bc1, Y_bc2, Z_bc1, Z_bc2])
        u_bc = jnp.vstack([u_bcx1, u_bcx2, u_bcy1, u_bcy2, u_bcz1, u_bcz2])

        u_pred = vmap(self.u_net,(None,0,0,0,0))(params,X_bc[:,0],X_bc[:,1],X_bc[:,2],X_bc[:,3])
        loss = jnp.mean((u_pred.flatten() - u_bc.flatten())**2)
        return loss


    @partial(jit,static_argnums = (0,))
    def res_loss(self,params,data_res):
        Xr,_ = data_res
        res = vmap(self.res_net,(None,0,0,0,0))(params,Xr[:,0],Xr[:,1],Xr[:,2],Xr[:,3])
        loss = jnp.mean(res**2)
        return loss

    @partial(jit,static_argnums=(0,))
    def loss_fun(self,params,data_ic,data_bcx1,data_bcx2,data_bcy1,data_bcy2,data_bcz1,data_bcz2,data_res):

        loss_ic = self.ic_loss(params,data_ic)
        loss_bc = self.bc_loss(params,data_bcx1,data_bcx2,data_bcy1,data_bcy2,data_bcz1,data_bcz2)
        loss_res = self.res_loss(params,data_res)
        loss = loss_ic + loss_bc + loss_res
        return loss

    @partial(jit,static_argnums=(0,))
    def train_step(self,i,opt_state,data_ic,data_bcx1,data_bcx2,data_bcy1,data_bcy2,data_bcz1,data_bcz2,data_res):
        params = self.get_params(opt_state)

        g = grad(self.loss_fun,0)(params,data_ic,data_bcx1,data_bcx2,data_bcy1,data_bcy2,data_bcz1,data_bcz2,data_res)
        return self.opt_update(i,g,opt_state)

    def train(self,nIter = 10000,batch_size = 128):
        pbar = trange(nIter)
        for it in pbar:

            if it%100 == 0:
                data_bcx1 = self.bcx_sampler1.sample(batch_size//6)
                data_bcx2 = self.bcx_sampler2.sample(batch_size//6)
                data_bcy1 = self.bcy_sampler1.sample(batch_size//6)
                data_bcy2 = self.bcy_sampler2.sample(batch_size//6)
                data_bcz1 = self.bcz_sampler1.sample(batch_size//6)
                data_bcz2 = self.bcz_sampler2.sample(batch_size//6)
                data_ic = self.ic_sampler.sample(batch_size)
                data_res = self.res_sampler.sample(batch_size)


            counter = next(self.count)
            self.opt_state = self.train_step(counter,self.opt_state,data_ic,data_bcx1,data_bcx2,
                                             data_bcy1,data_bcy2,data_bcz1,data_bcz2,data_res)

            if (it+1) % 100 == 0:
                params = self.get_params(self.opt_state)
                loss_ic = self.ic_loss(params,data_ic)
                loss_bc = self.bc_loss(params,data_bcx1,data_bcx2,
                                             data_bcy1,data_bcy2,data_bcz1,data_bcz2)
                loss_res = self.res_loss(params,data_res)
                self.hist_loss_res.append(loss_res)
                self.hist_loss_ic.append(loss_ic)
                self.hist_loss_bc.append(loss_bc)
                b = jax.nn.elu(params[-1],0)
                self.b_hist.append(b)
                pbar.set_postfix({'loss_ic':loss_ic,
                                'loss_bc':loss_bc,
                                'loss_res':loss_res,
                                'b':b
                                })




In [33]:

layers = [51,20,20,1]
mu0 = .01
delta2 = 1.1
kappa = .5
nu = 0.05
p = .2
theta = 0.05
C0 = 0.05
alpha = -delta2**2 + 2 * kappa

denom_part = alpha - 3 * p
numer_coeff = 6 * alpha * nu * delta2
A = numer_coeff / denom_part
def u(x,y,z,t):
    inner_arg = kappa * t + (3 * p * y) / alpha + p * z + theta + x + C0
    exp_term = jnp.exp(-delta2 * inner_arg)
    denom = exp_term + nu
    out = mu0 - (A / denom)
    return out

x = jnp.linspace(-20,20,101)
y_fixed = 0.5
z_fixed = 0.5
t = jnp.linspace(-10,10,101)

lb = jnp.array([-20.,-1.,-1.,-10])
ub = jnp.array([20,1.,1.,10])

coords = jnp.array([lb,ub])
coord_bcx1 = jnp.array([[-20,-1.,-1.,-10],[-20,1.,1.,10]])
coord_bcx2 = jnp.array([[20,-1.,-1.,-10],[20,1.,1.,10]])
coord_bcy1 = jnp.array([[-20,-1.,-1.,-10],[20,-1.,1.,10]])
coord_bcy2 = jnp.array([[-20,1.,-1.,-10],[20,1.,1.,10]])
coord_bcz1 = jnp.array([[-20,-1.,-1.,-10],[20,1.,-1.,10]])
coord_bcz2 = jnp.array([[-20,-1.,1.,-10],[20,1.,1.,10]])

coord_ic = jnp.array([[-20.,-1.,-1.,-10],[20.,1.,1.,-10]])

bcx_sampler1 = Sampler(4,coord_bcx1,u)
bcx_sampler2 = Sampler(4,coord_bcx2,u)
bcy_sampler1 = Sampler(4,coord_bcy1,u)
bcy_sampler2 = Sampler(4,coord_bcy2,u)
bcz_sampler1 = Sampler(4,coord_bcz1,u)
bcz_sampler2 = Sampler(4,coord_bcz2,u)
bc_sampler = [bcx_sampler1,bcx_sampler2,bcy_sampler1,bcy_sampler2,bcz_sampler1,bcz_sampler2]
ic_sampler = Sampler(4,coord_ic,u)

res_sampler = Sampler(4,coords,lambda x,y,z,t: 0*x+0*y+0*z+0*t)

In [ ]:
model = PINN(layers,bc_sampler,ic_sampler,res_sampler,X_train,u_train,2,lb,ub)
model.train(nIter=20000)

In [ ]:
model.hist_loss_res[-1],model.hist_loss_ic[-1],model.hist_loss_bc[-1]

In [ ]:
params = model.get_params(model.opt_state)
u_pred = vmap(vmap(model.u_net,(None,None,None,None,0)),(None,0,None,None,None))(params,x,0.5,0.5,t)
u_exact = vmap(vmap(u,(None,None,None,0)),(0,None,None,None))(x,0.5,0.5,t)
L2error = jnp.linalg.norm(u_pred[:,-1] - u_exact[:,-1])/jnp.linalg.norm(u_exact[:,-1])
L2error

In [ ]:
fig, ax = plt.subplots(1,2,subplot_kw={"projection": "3d"},figsize = (10,3))

surf1 = ax[0].plot_surface(X, T, u_exact.T, cmap='jet',
                       linewidth=0, antialiased=False)
ax[0].zaxis.set_major_formatter('{x:.01f}')
ax[0].view_init(azim=105, elev=20)
ax[0].set_xlabel(r'x')
ax[0].set_ylabel(r't')
ax[0].set_zlabel(r'q(x,0.5,0.5,t)')
ax[1].set_title('Predicted Dynamic')
surf2 = ax[1].plot_surface(X, T, u_pred.T, cmap='jet',
                       linewidth=0, antialiased=False)

ax[1].zaxis.set_major_formatter('{x:.01f}')
ax[1].view_init(azim=105, elev=20)
ax[1].set_xlabel(r'x')
ax[1].set_ylabel(r't')
ax[1].set_zlabel(r'$\tilde{q}(x,0.5,0.5,t)$')
ax[0].set_title('Exact Dynamic')
plt.tight_layout()
plt.subplots_adjust( wspace=-0.5)
#plt.savefig(fname = 'Dynamic2.pdf',format = 'pdf')
plt.show()

In [ ]:
fig,ax = plt.subplots(1,2,figsize = (10,3))
ax[0].pcolor(X,T,u_exact.T,cmap = 'jet')
ax[1].pcolor(X,T,u_pred.T,cmap = 'jet')
plt.show()

In [ ]:
fig,ax = plt.subplots(1,3,figsize = (14,4))
ax[0].plot(np.arange(0,30000,100),model.hist_loss_res,'b--',label = r'$Loss_{r}$',lw = 3)
ax[0].plot(np.arange(0,30000,100),model.hist_loss_bc,'r--',label = r'$Loss_{bc}$',lw = 3)
ax[0].plot(np.arange(0,30000,100),model.hist_loss_ic,'g--',label = r'$Loss_{ic}$',lw = 3)
ax[0].set_yscale('log')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss (log)')
ax[2].scatter(x,np.abs(u_exact[:,0] - u_pred[:,0]),facecolor = 'b',marker = 'x',label = r'$t = -0.1$')
ax[2].scatter(x,np.abs(u_exact[:,50] - u_pred[:,50]),facecolor = 'r',marker = 'x',label = r'$t = 0$')
ax[2].scatter(x,np.abs(u_exact[:,-1] - u_pred[:,-1]),facecolor = 'g',marker = 'x',label = r'$t = 0.1$')
ax[2].set_xlabel(r'x')
ax[2].set_ylabel(r'$|\tilde{q}(x,0.5,0.5,t) - q(x,0.5,0.5,t)|$')
ax[1].plot(np.arange(0,30000,100),model.b_hist,'k',lw = 3)
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel(r'$b$')
plt.tight_layout()
ax[0].legend(frameon = False)
ax[2].legend(frameon = False)
ax[0].grid()
ax[1].grid()
ax[2].grid()
#plt.savefig(fname = 'loss_error1.pdf',format = 'pdf')
plt.show()

In [ ]:
fig,ax = plt.subplots(1,3,figsize = (14,4))
ax[0].plot(x,u_exact[:,0],'b',lw = 3,label = 'Exact')
ax[0].plot(x,u_pred[:,0],'*',markeredgecolor = 'r', markeredgewidth = 1,label = 'Predicted', fillstyle='none')
ax[1].plot(x,u_exact[:,50],'b',lw = 3,label = 'Exact')
ax[1].plot(x,u_pred[:,50],'*',markeredgecolor = 'r', markeredgewidth = 1,label = 'Predicted', fillstyle='none')
ax[2].plot(x,u_exact[:,-1],'b',lw = 3,label = 'Exact')
ax[2].plot(x,u_pred[:,-1],'*',markeredgecolor = 'r', markeredgewidth = 1,label = 'Predicted', fillstyle='none')
ax[0].set_xlabel(r'$x$')
ax[1].set_xlabel(r'$x$')
ax[2].set_xlabel(r'$x$')
ax[0].set_ylabel(r'$q(x,0.5,0.5,t)$')
ax[0].set_title(r'$t = -10$')
ax[1].set_title(r'$t = 0$')
ax[2].set_title(r'$t = 10$')
ax[0].grid()
ax[1].grid()
ax[2].grid()
ax[0].legend(frameon = False)
ax[1].legend(frameon = False)
ax[2].legend(frameon = False)
#plt.savefig(fname = 'sol2.pdf',format = 'pdf')
plt.show()